In [ ]:
import numpy as np, pandas as pd

TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-09-01' 
VAL_START_DATE = '2020-09-01'
VAL_END_DATE = '2021-09-01'
TRADE_START_DATE = '2021-09-01' 
TRADE_END_DATE = '2026-09-07'



Validation_DAYS = 365

def cagr(r):
    total = (1 + r).prod()
    years = len(r) / Validation_DAYS
    return total ** (1 / years) - 1

def vola(r):
    return r.std() * np.sqrt(Validation_DAYS)

def sharpe(r, rf=0.0):
    return (cagr(r) - rf) / vola(r)

def sortino(r, rf=0.0):
    down = r[r < 0].std() * np.sqrt(Validation_DAYS)
    return (cagr(r) - rf) / down

def max_drawdown(r):
    equity = (1 + r).cumprod()
    return (equity / equity.cummax() - 1).min()

def calmar(r):
    return cagr(r) / abs(max_drawdown(r))

def kennzahlen(r, name):
    return {"Strategie": name, "CAGR": cagr(r), "Vola": vola(r),
            "Sharpe": sharpe(r), "Sortino": sortino(r),
            "MaxDD": max_drawdown(r), "Calmar": calmar(r)}

In [ ]:
import pandas as pd, numpy as np

trade = pd.read_csv('../data/trade.csv')
px = trade.pivot(index='date', columns='tic', values='close').sort_index()
px.index = pd.to_datetime(px.index)
px = px.replace(0, np.nan).ffill().dropna(axis=1, how='any')

In [ ]:
START, FEE = 25_000.0, 1.0

def equity(px, rebal_days, start=START, fee=FEE):
    """Depotwert je Handelstag. Rebalancing auf Gleichgewichtung an rebal_days."""
    n = px.shape[1]
    shares = pd.Series(0.0, index=px.columns)
    cash, out = start, []
    for d in px.index:
        p = px.loc[d]
        if d in rebal_days:
            value = cash + (shares * p).sum()
            ziel = (value / n) / p
            orders = int(((ziel - shares).abs() * p > 1.0).sum())   # Mindestorder 1 EUR
            value -= orders * fee
            shares, cash = (value / n) / p, 0.0
        out.append(cash + (shares * p).sum())
    return pd.Series(out, index=px.index)

monats_erste = px.groupby(px.index.to_period("M")).head(1).index

eq_bh_1n   = equity(px, rebal_days=[px.index[0]])        # 1/N gekauft und gehalten
eq_reb_1n  = equity(px, rebal_days=monats_erste)         # 1/N monatlich  ← der Gegner
eq_index   = equity(px[["EUNL.DE"]], rebal_days=[px.index[0]])

In [ ]:
from finrl.agents.stablebaselines3.models import DRLAgent

e_trade = TRStockTradingEnv(df=trade, fixed_fee=1.0, **env_tweeks)
df_account, df_actions = DRLAgent.DRL_prediction(model=trained_a2c, environment=e_trade)
eq_agent = df_account.set_index("date")["account_value"]

In [ ]:
serien = {"A2C": eq_agent, "1/N monatlich": eq_reb_1n,
          "1/N halten": eq_bh_1n, "MSCI World": eq_index}

tabelle = pd.DataFrame([
    kennzahlen(s.pct_change().dropna(), name) for name, s in serien.items()
])
tabelle.to_csv("results/runs.csv", index=False)